In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r'Dataset\extracted_dataset\non-multisegment\non-multisegment_input.csv')

df.head()

,filename,Event,EventTAG,LAT,LON,DEP,LEN_f,WID,Mw,Mo,...,HR-GPS-Data,Hr-GPS-Data,InSAR-Data,Other-Data,SAT-Data,SPOT-Data,hr-GPS-Data,inSAR-Data,level-Data,tril-Data
0,s1906SANFRA01SONG.fsp,San Francisco (Calif.),s1906SANFRA01SONG,37.78,-122.51,10.0,480.0,12.0,7.91,8.150000e+20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,s1906SANFRA01THAT.fsp,San Francisco (Calif.),s1906SANFRA01THAT,37.78,-122.51,10.0,480.0,10.0,7.91,8.120000e+20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,s1923KANTOJ01KOBA.fsp,Kanto (Japan),s1923KANTOJ01KOBA,35.40,139.20,14.6,130.0,70.0,8.08,1.460000e+21,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,s1923KANTOJ01WALD.fsp,Kanto (Japan),s1923KANTOJ01WALD,35.40,139.20,14.6,130.0,70.0,7.95,9.330000e+20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,s1944TONANK01ICHI.fsp,Tonankai (Japan),s1944TONANK01ICHI,33.77,135.96,30.0,220.0,140.0,8.04,1.310000e+21,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
import os

# Path to the folder containing the images
image_folder = r'Dataset\all_images'

# Check if the folder exists
if os.path.exists(image_folder):
    # Get all files in the folder
    image_files = [f[24:-4] for f in os.listdir(image_folder) if os.path.isfile(os.path.join(image_folder, f))]
    
    # Print the number of files found
    print(f"Found {len(image_files)} files in {image_folder} folder")
    
    # Display the first few filenames
    if image_files:
        print("First 5 filenames:")
        for file in image_files[:5]:
            print(file)
else:
    print(f"The folder '{image_folder}' does not exist.")

Found 200 files in Dataset\all_images folder
First 5 filenames:
s1923KANTOJ01KOBA.fsp
s1923KANTOJ01WALD.fsp
s1944TONANK01ICHI.fsp
s1944TONANK01KATO.fsp
s1944TONANK01KIKU.fsp


In [4]:
# Create a map for Dz values and save as JSON
import json

# Initialize the map for Dz values
dz_map = {}

# Select the columns we need (filename and Dz)
dz_cols = ['filename', 'Dz']

# Check if Dz column exists in the dataframe
if 'Dz' in df.columns:
    for row in df[dz_cols].iterrows():
        filename = row[1].values[0]  # Keep the .fsp extension for consistency
        dz_value = row[1].values[1]
        
        # Only add if Dz value is not NaN
        if not pd.isna(dz_value):
            dz_map[filename[:-4]] = float(dz_value)
    
    # Save the map as JSON
    with open('dz.json', 'w') as f:
        json.dump(dz_map, f, indent=2)
    
    print(f"Created dz.json with {len(dz_map)} entries")
    print(f"Sample entries: {dict(list(dz_map.items())[:5])}")
else:
    print("Warning: 'Dz' column not found in the dataframe")
    print(f"Available columns: {df.columns.tolist()}")



Created dz.json with 355 entries
Sample entries: {'s1906SANFRA01SONG': 12.0, 's1906SANFRA01THAT': 10.0, 's1923KANTOJ01KOBA': 10.0, 's1923KANTOJ01WALD': 10.0, 's1944TONANK01ICHI': 20.0}


In [5]:
df_cleaned = df[df['filename'].isin(image_files)]

df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 200 entries, 2 to 350
Data columns (total 64 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   filename     200 non-null    object 
 1   Event        200 non-null    object 
 2   EventTAG     200 non-null    object 
 3   LAT          200 non-null    float64
 4   LON          200 non-null    float64
 5   DEP          200 non-null    float64
 6   LEN_f        200 non-null    float64
 7   WID          200 non-null    float64
 8   Mw           200 non-null    float64
 9   Mo           200 non-null    float64
 10  STRK         200 non-null    float64
 11  DIP          200 non-null    float64
 12  RAKE         200 non-null    float64
 13  Htop         200 non-null    float64
 14  HypX         200 non-null    float64
 15  HypZ         200 non-null    float64
 16  avTr         200 non-null    float64
 17  avVr         200 non-null    float64
 18  Nx           200 non-null    float64
 19  Nz           

In [10]:
df[df['avVr'] == 999]['avTr'].value_counts()

avTr
999.0    96
11.1      1
9.0       1
31.1      1
8.4       1
90.3      1
22.0      1
44.6      1
Name: count, dtype: int64

In [11]:
selected_cols = ['filename', 'LEN_f', 'WID', 'RAKE', 'HypX', 'HypZ', 'Htop', 'DIP', "STRK", "Mo"]

map = {}

for row in df[selected_cols].iterrows():
    filename = row[1].values[0][:-4]
    if filename not in map:
        map[filename] = row[1].values[1:]

In [ ]:
map

In [13]:
import numpy as np

# Save the map as a .npy file
np.save(r'Dataset/text_vec.npy', map)
print(f"Saved text_vec.npy with {len(map)} entries")


Saved text_vec.npy with 355 entries
